# DistilBERT Experiment 07 - Train + Export + Demo Inference

This notebook uses the recommended Experiment C settings for presentation-quality results, then exports a portable model bundle for local demo inference.

## What this notebook does
- Trains DistilBERT multilabel toxicity model with fixed settings
- Tunes per-label thresholds on validation set
- Exports model/tokenizer/thresholds/labels/config
- Shows demo inference code that can run in another environment

In [ ]:
# Install deps in Colab (safe to rerun)
!pip -q install torch transformers pandas numpy scikit-learn

In [ ]:
# Paths and environment
from pathlib import Path
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

if IN_COLAB:
    # CHANGE if needed
    PROJECT_ROOT = Path('/content/drive/MyDrive/cmpe258/jigsaw-toxic-comment-classification-challenge')
else:
    # Resolve repo root robustly even if notebook is nested in subfolders.
    PROJECT_ROOT = Path.cwd().resolve()
    for c in (PROJECT_ROOT, PROJECT_ROOT.parent, PROJECT_ROOT.parent.parent, PROJECT_ROOT.parent.parent.parent):
        if (c / 'preprocessing' / 'text_preprocessing.py').exists():
            PROJECT_ROOT = c
            break

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'PROJECT_ROOT not found: {PROJECT_ROOT}')

NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
OUTPUT_DIR = PROJECT_ROOT / 'output'
ARTIFACT_DIR = NOTEBOOKS_DIR / 'artifacts_colab_exp_07'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(NOTEBOOKS_DIR))

print('PROJECT_ROOT:', PROJECT_ROOT)
print('ARTIFACT_DIR:', ARTIFACT_DIR)

In [ ]:
import contextlib
import json
import math
import shutil
import time

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

from preprocessing.text_preprocessing import LABEL_COLUMNS, preprocess_for_distilbert
from metrics_helpers import multilabel_evaluation_report, torch_parameter_count

In [ ]:
# Recommended Experiment C settings for demo
SEED = 42
TRAIN_SIZE = 130_000

MAX_LENGTH = 192
BATCH_SIZE = 24
MAX_EPOCHS = 6
LR = 1.5e-5
WEIGHT_DECAY = 0.02
WARMUP_RATIO = 0.15

EARLY_STOP_PATIENCE = 3
EARLY_STOP_MIN_DELTA = 0.001
MAX_GRAD_NORM = 1.0
GRADIENT_ACCUMULATION_STEPS = 2

USE_BF16 = True
USE_TORCH_COMPILE = False
NUM_WORKERS = 2
THRESHOLD_GRID = np.arange(0.05, 0.951, 0.01)

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_bf16_ok = DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported()
USE_AMP = bool(USE_BF16 and _bf16_ok)
if USE_BF16 and DEVICE.type == 'cuda' and not _bf16_ok:
    print('bf16 requested but not supported; using full precision.')

print('Device:', DEVICE)
print('AMP (bf16):', USE_AMP)

In [ ]:
def binary_f1(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    denom = 2 * tp + fp + fn
    return float((2 * tp) / denom) if denom > 0 else 0.0


def tune_per_label_thresholds(y_true: np.ndarray, y_prob: np.ndarray, labels, threshold_grid: np.ndarray):
    best_thresholds = {}
    rows = []
    for j, label in enumerate(labels):
        y_true_j = y_true[:, j]
        y_prob_j = y_prob[:, j]
        best_t = 0.5
        best_f1 = -1.0
        for t in threshold_grid:
            y_pred_j = (y_prob_j >= t).astype(np.int64)
            f1_j = binary_f1(y_true_j, y_pred_j)
            if f1_j > best_f1:
                best_f1 = f1_j
                best_t = float(t)
        best_thresholds[label] = best_t
        rows.append({'label': label, 'best_threshold': round(best_t, 3), 'best_f1_on_val': best_f1})
    return best_thresholds, pd.DataFrame(rows)


def apply_thresholds(y_prob: np.ndarray, labels, thresholds: dict[str, float]) -> np.ndarray:
    out = np.zeros_like(y_prob, dtype=np.int64)
    for j, label in enumerate(labels):
        out[:, j] = (y_prob[:, j] >= thresholds[label]).astype(np.int64)
    return out


def compute_pos_weight(y_train: np.ndarray, eps: float = 1e-6, max_weight: float = 50.0) -> torch.Tensor:
    y = np.asarray(y_train, dtype=np.float32)
    positives = y.sum(axis=0)
    total = float(y.shape[0])
    negatives = total - positives
    weights = negatives / np.maximum(positives, eps)
    weights = np.clip(weights, 1.0, max_weight)
    return torch.tensor(weights, dtype=torch.float32)


class EncDataset(Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item['labels'] = self.labels[i]
        return item


def collate(batch):
    out = {k: torch.stack([b[k] for b in batch], dim=0) for k in batch[0] if k != 'labels'}
    out['labels'] = torch.stack([b['labels'] for b in batch], dim=0)
    return out


def enc_dict(enc):
    keys = [k for k in ('input_ids', 'attention_mask') if k in enc]
    return {k: enc[k] for k in keys}

In [ ]:
# Load data/tokenizer using the existing preprocessing pipeline
run_data = preprocess_for_distilbert(
    validation_fraction=0.1,
    random_state=SEED,
    max_length=MAX_LENGTH,
    return_tensors='pt',
    max_train_samples=TRAIN_SIZE,
    max_val_samples=None,
)

tokenizer = run_data.tokenizer
train_enc = enc_dict(run_data.train_encodings)
val_enc = enc_dict(run_data.val_encodings)
y_train = np.asarray(run_data.y_train, dtype=np.float32)
y_val = np.asarray(run_data.y_val, dtype=np.int64)

print('train samples:', y_train.shape[0])
print('val samples:', y_val.shape[0])
print('labels:', LABEL_COLUMNS)

In [ ]:
# Build loaders/model/optimizer
pin = DEVICE.type == 'cuda'

train_loader = DataLoader(
    EncDataset(train_enc, y_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate,
    num_workers=NUM_WORKERS,
    pin_memory=pin,
    persistent_workers=NUM_WORKERS > 0,
)
val_loader = DataLoader(
    EncDataset(val_enc, y_val),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate,
    num_workers=NUM_WORKERS,
    pin_memory=pin,
    persistent_workers=NUM_WORKERS > 0,
)

model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=len(LABEL_COLUMNS),
    problem_type='multi_label_classification',
).to(DEVICE)

if USE_TORCH_COMPILE and hasattr(torch, 'compile'):
    try:
        model = torch.compile(model)
        print('torch.compile enabled')
    except Exception as e:
        print('torch.compile skipped:', e)

num_params = torch_parameter_count(model)
pos_weight = compute_pos_weight(y_train).to(DEVICE)

no_decay = ['bias', 'LayerNorm.weight']
optimizer_grouped_parameters = [
    {
        'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
        'weight_decay': WEIGHT_DECAY,
    },
    {
        'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
        'weight_decay': 0.0,
    },
]
optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=LR)
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

steps_per_epoch = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS)
num_training_steps = steps_per_epoch * MAX_EPOCHS
warmup_steps = int(WARMUP_RATIO * num_training_steps)
warmup_steps = max(0, min(warmup_steps, num_training_steps - 1)) if num_training_steps > 0 else 0
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=num_training_steps
)

if USE_AMP:
    autocast_ctx = torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=True)
else:
    autocast_ctx = contextlib.nullcontext()

print('num_parameters:', num_params)
print('steps_per_epoch:', steps_per_epoch)
print('warmup_steps:', warmup_steps)

In [ ]:
# Train with early stopping and keep best checkpoint in memory
train_time_s = 0.0
inference_time_s = 0.0
train_loss_last = float('nan')
val_loss_last = float('nan')

best_val_loss = float('inf')
best_state_cpu = None
best_epoch = 0
patience_left = EARLY_STOP_PATIENCE
epochs_ran = 0
early_stopped = False

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_train_loss = 0.0
    epoch_batches = 0
    t0 = time.perf_counter()
    optimizer.zero_grad(set_to_none=True)
    micro = 0

    for batch in train_loader:
        batch = {k: v.to(DEVICE, non_blocking=pin) for k, v in batch.items()}
        labels = batch.pop('labels')
        with autocast_ctx:
            logits = model(**batch).logits
            loss = loss_fn(logits, labels) / GRADIENT_ACCUMULATION_STEPS
        loss.backward()
        micro += 1
        epoch_train_loss += float(loss.item()) * GRADIENT_ACCUMULATION_STEPS
        epoch_batches += 1

        if micro % GRADIENT_ACCUMULATION_STEPS == 0:
            if MAX_GRAD_NORM is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

    if micro % GRADIENT_ACCUMULATION_STEPS != 0:
        if MAX_GRAD_NORM is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

    train_time_s += time.perf_counter() - t0
    train_loss_last = epoch_train_loss / max(epoch_batches, 1)

    model.eval()
    val_loss_sum = 0.0
    val_batches = 0
    t1 = time.perf_counter()
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(DEVICE, non_blocking=pin) for k, v in batch.items()}
            labels = batch.pop('labels')
            with autocast_ctx:
                logits = model(**batch).logits
                val_loss_sum += float(loss_fn(logits, labels).item())
            val_batches += 1
    inference_time_s += time.perf_counter() - t1
    val_loss_last = val_loss_sum / max(val_batches, 1)
    epochs_ran = epoch

    print(f'epoch {epoch}/{MAX_EPOCHS} train_loss={train_loss_last:.4f} val_loss={val_loss_last:.4f} lr={scheduler.get_last_lr()[0]:.2e}')

    improved = val_loss_last < (best_val_loss - EARLY_STOP_MIN_DELTA)
    if improved:
        best_val_loss = val_loss_last
        best_epoch = epoch
        best_state_cpu = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        patience_left = EARLY_STOP_PATIENCE
    else:
        patience_left -= 1
        if patience_left <= 0:
            early_stopped = True
            print(f'early stopping at epoch {epoch} (best_epoch={best_epoch}, best_val_loss={best_val_loss:.4f})')
            break

if best_state_cpu is not None:
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state_cpu.items()})

print('best_epoch:', best_epoch)
print('best_val_loss:', best_val_loss)
print('epochs_ran:', epochs_ran)
print('early_stopped:', early_stopped)

In [ ]:
# Validate + tune thresholds
model.eval()
probs_all = []
val_loss_sum = 0.0
val_batches = 0

with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(DEVICE, non_blocking=pin) for k, v in batch.items()}
        labels = batch.pop('labels')
        with autocast_ctx:
            logits = model(**batch).logits
            probs = torch.sigmoid(logits)
            val_loss_sum += float(loss_fn(logits, labels).item())
        probs_all.append(probs.float().cpu().numpy())
        val_batches += 1

val_loss = val_loss_sum / max(val_batches, 1)
prob_val = np.concatenate(probs_all, axis=0)
pred_05 = (prob_val >= 0.5).astype(np.int64)

per_label_05_df, summary_05 = multilabel_evaluation_report(y_val, pred_05, prob_val, LABEL_COLUMNS)
best_thresholds, thresholds_df = tune_per_label_thresholds(y_val, prob_val, LABEL_COLUMNS, THRESHOLD_GRID)
pred_tuned = apply_thresholds(prob_val, LABEL_COLUMNS, best_thresholds)
per_label_tuned_df, summary_tuned = multilabel_evaluation_report(y_val, pred_tuned, prob_val, LABEL_COLUMNS)

print('val_loss:', round(val_loss, 6))
print('f1_micro_baseline:', round(summary_05['f1_micro'], 6))
print('f1_macro_baseline:', round(summary_05['f1_macro'], 6))
print('f1_micro_tuned:', round(summary_tuned['f1_micro'], 6))
print('f1_macro_tuned:', round(summary_tuned['f1_macro'], 6))

display(thresholds_df)

In [ ]:
# Export a portable demo bundle
export_dir = ARTIFACT_DIR / 'distilbert_exp07_demo_bundle'
export_dir.mkdir(parents=True, exist_ok=True)

# Attach label metadata to config for easier downstream loading
model.config.id2label = {i: label for i, label in enumerate(LABEL_COLUMNS)}
model.config.label2id = {label: i for i, label in enumerate(LABEL_COLUMNS)}

model_to_save = model.module if hasattr(model, 'module') else model
model_to_save.save_pretrained(export_dir)
tokenizer.save_pretrained(export_dir)

with open(export_dir / 'thresholds.json', 'w') as f:
    json.dump(best_thresholds, f, indent=2)

with open(export_dir / 'labels.json', 'w') as f:
    json.dump(list(LABEL_COLUMNS), f, indent=2)

demo_config = {
    'train_size': TRAIN_SIZE,
    'max_length': MAX_LENGTH,
    'batch_size': BATCH_SIZE,
    'max_epochs': MAX_EPOCHS,
    'lr': LR,
    'weight_decay': WEIGHT_DECAY,
    'warmup_ratio': WARMUP_RATIO,
    'early_stop_patience': EARLY_STOP_PATIENCE,
    'early_stop_min_delta': EARLY_STOP_MIN_DELTA,
    'grad_clip': MAX_GRAD_NORM,
    'grad_accum': GRADIENT_ACCUMULATION_STEPS,
    'use_bf16_amp': USE_AMP,
    'num_workers': NUM_WORKERS,
    'best_epoch': best_epoch,
    'best_val_loss': best_val_loss,
    'train_time_s': train_time_s,
    'inference_time_s': inference_time_s,
    'f1_micro_tuned': summary_tuned['f1_micro'],
    'f1_macro_tuned': summary_tuned['f1_macro'],
}
with open(export_dir / 'demo_config.json', 'w') as f:
    json.dump(demo_config, f, indent=2)

thresholds_df.to_csv(export_dir / 'thresholds_table.csv', index=False)
per_label_tuned_df.to_csv(export_dir / 'per_label_tuned_metrics.csv', index=False)

zip_base = ARTIFACT_DIR / 'distilbert_exp07_demo_bundle'
zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir=export_dir)

print('Export folder:', export_dir)
print('Zip file:', zip_path)
print('Files:')
for p in sorted(export_dir.glob('*')):
    print(' -', p.name)

## Demo inference in another notebook/script

Use the exported bundle folder (`distilbert_exp07_demo_bundle`) on your local machine. The following cell is standalone and can run outside Colab.

In [ ]:
# Standalone demo inference (can run in a different local notebook)
import json
from pathlib import Path

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# IMPORTANT: point this at the extracted export folder (not the .zip file)
BUNDLE_DIR = Path('./distilbert_exp07_demo_bundle').expanduser().resolve()

if not BUNDLE_DIR.exists() or not BUNDLE_DIR.is_dir():
    raise FileNotFoundError(
        f'Bundle folder not found: {BUNDLE_DIR}\n'
        'Set BUNDLE_DIR to the extracted distilbert_exp07_demo_bundle folder.'
    )

required_files = ['config.json', 'thresholds.json', 'labels.json']
missing = [name for name in required_files if not (BUNDLE_DIR / name).exists()]
if missing:
    raise FileNotFoundError(
        f'Missing required files in bundle: {missing}\n'
        f'Found folder: {BUNDLE_DIR}'
    )

tokenizer_demo = AutoTokenizer.from_pretrained(str(BUNDLE_DIR), local_files_only=True)
model_demo = AutoModelForSequenceClassification.from_pretrained(str(BUNDLE_DIR), local_files_only=True)
model_demo.eval()

with open(BUNDLE_DIR / 'thresholds.json', 'r') as f:
    thresholds_demo = json.load(f)
with open(BUNDLE_DIR / 'labels.json', 'r') as f:
    labels_demo = json.load(f)


def predict_toxicity(texts: list[str]):
    enc = tokenizer_demo(texts, padding=True, truncation=True, max_length=192, return_tensors='pt')
    with torch.no_grad():
        logits = model_demo(**enc).logits
        probs = torch.sigmoid(logits).cpu().numpy()

    rows = []
    for i, text in enumerate(texts):
        row = {'text': text}
        for j, label in enumerate(labels_demo):
            p = float(probs[i, j])
            t = float(thresholds_demo[label])
            row[f'{label}_prob'] = round(p, 4)
            row[f'{label}_pred'] = int(p >= t)
        rows.append(row)
    return rows

sample_texts = [
    'Thanks for helping me with this assignment.',
    'You are a complete idiot and nobody likes you.',
    'I disagree with your point, but happy to discuss politely.'
]

results = predict_toxicity(sample_texts)
for r in results:
    print('-' * 80)
    print(r['text'])
    for label in labels_demo:
        print(f"{label:14s} prob={r[f'{label}_prob']:.4f} pred={r[f'{label}_pred']}")

In [ ]:
# Optional: write a reusable local demo script
script_path = OUTPUT_DIR / 'distilbert_exp07_demo_inference.py'
script_text = '''import argparse
import json
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification


def load_bundle(bundle_dir: str):
    tokenizer = AutoTokenizer.from_pretrained(bundle_dir)
    model = AutoModelForSequenceClassification.from_pretrained(bundle_dir)
    model.eval()
    with open(f"{bundle_dir}/thresholds.json", "r") as f:
        thresholds = json.load(f)
    with open(f"{bundle_dir}/labels.json", "r") as f:
        labels = json.load(f)
    return tokenizer, model, thresholds, labels


def predict(texts, tokenizer, model, thresholds, labels, max_length=192):
    enc = tokenizer(texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
    with torch.no_grad():
        probs = torch.sigmoid(model(**enc).logits).cpu().numpy()

    outputs = []
    for i, text in enumerate(texts):
        row = {"text": text}
        for j, label in enumerate(labels):
            p = float(probs[i, j])
            row[f"{label}_prob"] = p
            row[f"{label}_pred"] = int(p >= float(thresholds[label]))
        outputs.append(row)
    return outputs


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--bundle_dir", required=True)
    parser.add_argument("--text", action="append", required=True, help="Pass multiple --text entries")
    args = parser.parse_args()

    tok, model, thr, labels = load_bundle(args.bundle_dir)
    preds = predict(args.text, tok, model, thr, labels)
    print(json.dumps(preds, indent=2))
'''
script_path.write_text(script_text)
print('Wrote script:', script_path)
print('Example:')
print('python3 output/distilbert_exp07_demo_inference.py --bundle_dir ./distilbert_exp07_demo_bundle --text "you are great" --text "you are stupid"')